# HyperSIGMA native-geometry ablation (single-branch, unadapted)

Evaluates the **frozen** HyperSIGMA SpatViT / SpecViT branches **separately** at their
**native pretrained input geometry**, instead of shrinking the encoder to the 11x11 patch.

* **Native geometry** — SpatViT is built at `64x64 / patch-8 / 100ch` and SpecViT at `64x64`,
  so the pretrained `patch_embed.proj`, `spat_map`, and `pos_embed` **load from the checkpoint**
  (no reinit). The 11x11 patch is transformed *up to* 64x64 two ways: **upscale** (interpolate)
  and **pad** (centered, zero-fill).
* **Spatial channels** — the pretrained SpatViT `patch_embed` needs 100 channels. Houston (144
  bands) uses `PCA 144->100`; Trento (63) / MUUFL (64) cannot reach 100 via PCA so the dual
  spectrally **resamples** raw bands -> 100 on the fly.
* **Unadapted, single-branch only** — adapted checkpoints are shape-incompatible with the native
  geometry, so `adapted_checkpoint='none'` is hard-wired and SEM fusion is out of scope.

Grid: `3 datasets x {spatial, spectral} x {upscale, pad}` = **12 evaluations**, each written to
`experiments/<experiment_name>/evaluations/native_<ds>_<branch>_<fit>/`.

> The existing `notebooks/evaluate_hypersigma.ipynb` (adapt pipeline) is untouched; this is a
> standalone driver. Per project convention the GPU eval cells are run by **you**, not the agent.


In [2]:
# === Parameters ===
# Repo-root bootstrap so we can read per-dataset specs from the central registry.
import os, sys
from pathlib import Path as _Path

_REPO = _Path.cwd()
while not (_REPO / "lib" / "experiments.py").exists():
    if _REPO.parent == _REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    _REPO = _REPO.parent
os.chdir(_REPO)
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from data.datasets import get_spec

# --------------------------------------------------------------------------
# Ablation grid.
# --------------------------------------------------------------------------
datasets = ["houston", "trento", "muufl"]   # all three (drop entries to subset)
# (display name, HyperSIGMAFewShot eval mode) -- native is single-branch only.
branches = [("spatial", "spat_pool"), ("spectral", "spec_pool")]
input_fits = ["upscale", "pad"]             # 11x11 -> 64x64 strategies

# Few-shot episode shape (C-way: all classes, like evaluate_hypersigma.ipynb).
k_shot = 5
k_query = 100
num_episodes = 2000

# Umbrella experiment all 12 evals are logged under.
experiment_name = "hypersigma_native_ablation_run1"

# Native geometry is unadapted-only.
adapted_checkpoint = "none"

eval_device = "cuda:1"   # which GPU the evaluation runs on
overwrite = True

# Native spatial channel target: the pretrained SpatViT patch_embed expects 100ch.
NATIVE_SPAT_IN_CHANS = 100

# Base eval params shared by every run. Dataset-coupled paths (PCA, native PCA)
# are resolved per-dataset inside the eval script, so they are intentionally
# omitted here. `dataset`, `mode`, and `input_fit` are set per loop iteration.
base_eval_params = {
    "k_shot": k_shot,
    "k_query": k_query,
    "num_episodes": num_episodes,
    "split": "all",
    "seed": 42,
    "device": eval_device,
    "spat_ckpt": "checkpoints/hypersigma/spat-vit-base.pth",
    "spec_ckpt": "checkpoints/hypersigma/spec-vit-base.pth",
    "distance_metric": "euclidean",   # "euclidean" | "cosine"
    "temperature": 10.0,
    "prototype_mode": "mean_features",
    "num_example_episodes": 1,
    "max_tsne_samples": 100,
    "no_plots": False,
    "native_geometry": True,
}

print("Native-geometry ablation grid:")
for ds in datasets:
    s = get_spec(ds)
    print(f"  {ds:8s} bands={s.hsi_channels:3d} classes={s.num_classes:2d} "
          f"-> spatial: {'PCA 144->100' if s.hsi_channels >= NATIVE_SPAT_IN_CHANS else f'resample {s.hsi_channels}->100'}")
print(f"\n{len(datasets)} x {len(branches)} x {len(input_fits)} = "
      f"{len(datasets)*len(branches)*len(input_fits)} evaluations")
print(f"experiment_name = {experiment_name}")


Native-geometry ablation grid:
  houston  bands=144 classes=15 -> spatial: PCA 144->100
  trento   bands= 63 classes= 6 -> spatial: resample 63->100
  muufl    bands= 64 classes=11 -> spatial: resample 64->100

3 x 2 x 2 = 12 evaluations
experiment_name = hypersigma_native_ablation_run1


In [3]:
# Repo root + autoreload + torch/CUDA info.
import os, sys
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip is not None:
        ip.run_line_magic("load_ext", "autoreload")
        ip.run_line_magic("autoreload", "2")
        print("autoreload : enabled")
except Exception as exc:
    print(f"autoreload : skipped ({exc})")

import torch
print(f"Repo root : {REPO}")
print(f"torch     : {torch.__version__}")
print(f"CUDA      : available={torch.cuda.is_available()}, device_count={torch.cuda.device_count()}")


autoreload : enabled
Repo root : /work/nmaric/CoFFE/CoFFE
torch     : 2.11.0+cu128
CUDA      : available=True, device_count=4


## Step 1 - Checkpoints present

The native branches load the same upstream MAE checkpoints as the adapt pipeline. If missing,
run `bash scripts/download_hypersigma_checkpoints.sh checkpoints/hypersigma` first.


In [4]:
from pathlib import Path

required = {
    "spat-vit-base.pth": base_eval_params["spat_ckpt"],
    "spec-vit-base.pth": base_eval_params["spec_ckpt"],
}
missing = []
for name, path in required.items():
    p = Path(path)
    if p.exists():
        print(f"  [OK]      {p}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        missing.append(name)
        print(f"  [MISSING] {p}")
if missing:
    raise FileNotFoundError(
        f"Missing {missing}. Run: bash scripts/download_hypersigma_checkpoints.sh checkpoints/hypersigma"
    )


  [OK]      checkpoints/hypersigma/spat-vit-base.pth  (1436.4 MB)
  [OK]      checkpoints/hypersigma/spec-vit-base.pth  (1391.8 MB)


## Step 2 - Native spatial PCA (144->100) for datasets with >=100 bands

The pretrained SpatViT `patch_embed` expects 100 channels. Houston (144 bands) is reduced with a
`PCA 144->100` fitted here (CPU, one-off) and cached at `checkpoints/hypersigma/pca_houston_100band.pkl`
(+ `_stats.pkl` for input standardization). Trento (63) / MUUFL (64) have < 100 bands, so PCA cannot
reach 100 components - the dual spectrally resamples raw bands -> 100 on the fly (no fit needed).

Equivalent CLI: `python scripts/fit_pca_hypersigma.py --dataset houston --n-components 100`.


In [5]:
from pathlib import Path
from data.datasets import get_spec, DATASET_REGISTRY
from models.hypersigma.preprocessing import fit_dataset_pca, fit_pca_output_stats, load_pca

PCA_DIR = Path("checkpoints/hypersigma")

for ds in datasets:
    spec = get_spec(ds)
    if spec.hsi_channels < NATIVE_SPAT_IN_CHANS:
        print(f"{ds:8s}: {spec.hsi_channels} bands < {NATIVE_SPAT_IN_CHANS} "
              f"-> spectral resample at eval time (no PCA fit).")
        continue
    pca_path = PCA_DIR / f"pca_{ds}_{NATIVE_SPAT_IN_CHANS}band.pkl"
    stats_path = PCA_DIR / f"pca_{ds}_{NATIVE_SPAT_IN_CHANS}band_stats.pkl"
    if pca_path.exists():
        pca = load_pca(str(pca_path))
        print(f"{ds:8s}: found {pca_path} (components={pca.components_.shape})")
        continue
    print(f"{ds:8s}: fitting PCA {spec.hsi_channels}->{NATIVE_SPAT_IN_CHANS} (CPU) ...")
    dataset = spec.patched_cls(data_root="./data/raw", patch_size=11, split="all", normalize=True)
    pca = fit_dataset_pca(dataset=dataset, n_components=NATIVE_SPAT_IN_CHANS, save_path=str(pca_path))
    fit_pca_output_stats(dataset=dataset, pca=pca, save_path=str(stats_path))
    print(f"{ds:8s}: saved {pca_path} + {stats_path}")


houston : found checkpoints/hypersigma/pca_houston_100band.pkl (components=(100, 144))
trento  : 63 bands < 100 -> spectral resample at eval time (no PCA fit).
muufl   : 64 bands < 100 -> spectral resample at eval time (no PCA fit).


/work/nmaric/CoFFE/CoFFE/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Step 3 - Bootstrap the umbrella experiment dir


In [6]:
from lib.experiments import ExperimentLogger
logger_ = ExperimentLogger(repo_root=REPO)
try:
    logger_.get_experiment(experiment_name)
    print(f"Experiment '{experiment_name}' already exists")
except FileNotFoundError:
    logger_.start_pretrain(
        name=experiment_name,
        description=(
            "HyperSIGMA native-geometry ablation: frozen SpatViT (64x64/patch-8/100ch) and "
            "SpecViT (64x64) evaluated separately, unadapted, with 11x11 inputs upscaled or "
            "zero-padded up to the native size."
        ),
        config={"meta": "native-geometry single-branch ablation; unadapted"},
    )
    print(f"Created experiment '{experiment_name}'")


Experiment 'hypersigma_native_ablation_run1' already exists


## Step 4 - Run the grid (datasets x branches x input-fits)

Each run dispatches to `scripts.evaluate_hypersigma.run_evaluation` with `native_geometry=True`
and `adapted_checkpoint='none'`. The per-run `eval.log` prints the load summary - confirm
`patch_embed.proj.weight.shape=(768, 100, 8, 8)` / `spat_map.weight.shape=(768, 4096)` and
`pos_embed source=loaded`.


In [ ]:
import json
from lib.eval_runner import run_hypersigma_evaluation

runs = []
for ds in datasets:
    for branch_name, eval_mode in branches:
        for fit in input_fits:
            eval_name = f"native_{ds}_{branch_name}_{fit}"
            params = {**base_eval_params, "dataset": ds, "mode": eval_mode, "input_fit": fit}
            print(f"\n{'='*70}\n>>> {eval_name}\n{'='*70}")
            ev = run_hypersigma_evaluation(
                experiment_name=experiment_name,
                eval_name=eval_name,
                adapted_checkpoint=adapted_checkpoint,
                eval_params=params,
                overwrite=overwrite,
            )
            res = json.loads(ev.results_path.read_text())
            runs.append({"dataset": ds, "branch": branch_name, "fit": fit,
                         "eval_name": eval_name, "root": str(ev.root), "results": res})
            print(f"    done -> {ev.root}")
print(f"\nCompleted {len(runs)} runs.")


2026-06-06 23:00:38,163 - Starting HyperSIGMA eval 'native_houston_spatial_upscale' for experiment 'hypersigma_native_ablation_run1'
2026-06-06 23:00:38,164 - Using device: cuda:1
2026-06-06 23:00:38,164 - Resolved paths: pca_spat=checkpoints/hypersigma/pca_houston_3band.pkl pca_stats=checkpoints/hypersigma/pca_houston_3band_stats.pkl adapted=none



>>> native_houston_spatial_upscale


2026-06-06 23:00:40,426 - Loaded houston (all): 15029 samples
2026-06-06 23:00:41,870 - [HyperSIGMA] SpatViT load summary: dropped=106, missing=16 (post-reinit), unexpected=0, pos_embed=loaded
2026-06-06 23:00:41,871 - [HyperSIGMA] SpatViT missing keys after re-init (first 10): ['blocks.0.attn.sampling_offsets.weight', 'blocks.0.attn.sampling_offsets.bias', 'blocks.1.attn.sampling_offsets.weight', 'blocks.1.attn.sampling_offsets.bias', 'blocks.3.attn.sampling_offsets.weight', 'blocks.3.attn.sampling_offsets.bias', 'blocks.4.attn.sampling_offsets.weight', 'blocks.4.attn.sampling_offsets.bias', 'blocks.6.attn.sampling_offsets.weight', 'blocks.6.attn.sampling_offsets.bias']
2026-06-06 23:00:41,872 - [HyperSIGMA] Built branches: spat=True, spec=False, sem=False
2026-06-06 23:00:41,873 - [HyperSIGMA] Native-geometry ablation: input_fit=upscale (single-branch, unadapted)
2026-06-06 23:00:41,873 - [HyperSIGMA] Spatial PCA cumulative explained variance: 144 -> 100 : 1.0000
2026-06-06 23:00:41,

    done -> /work/nmaric/CoFFE/CoFFE/experiments/hypersigma_native_ablation_run1/evaluations/native_houston_spatial_upscale

>>> native_houston_spatial_pad


2026-06-06 23:30:36,922 - Loaded houston (all): 15029 samples
2026-06-06 23:30:37,974 - [HyperSIGMA] SpatViT load summary: dropped=106, missing=16 (post-reinit), unexpected=0, pos_embed=loaded
2026-06-06 23:30:37,975 - [HyperSIGMA] SpatViT missing keys after re-init (first 10): ['blocks.0.attn.sampling_offsets.weight', 'blocks.0.attn.sampling_offsets.bias', 'blocks.1.attn.sampling_offsets.weight', 'blocks.1.attn.sampling_offsets.bias', 'blocks.3.attn.sampling_offsets.weight', 'blocks.3.attn.sampling_offsets.bias', 'blocks.4.attn.sampling_offsets.weight', 'blocks.4.attn.sampling_offsets.bias', 'blocks.6.attn.sampling_offsets.weight', 'blocks.6.attn.sampling_offsets.bias']
2026-06-06 23:30:37,976 - [HyperSIGMA] Built branches: spat=True, spec=False, sem=False
2026-06-06 23:30:37,977 - [HyperSIGMA] Native-geometry ablation: input_fit=pad (single-branch, unadapted)
2026-06-06 23:30:37,977 - [HyperSIGMA] Spatial PCA cumulative explained variance: 144 -> 100 : 1.0000
2026-06-06 23:30:37,978 

    done -> /work/nmaric/CoFFE/CoFFE/experiments/hypersigma_native_ablation_run1/evaluations/native_houston_spatial_pad

>>> native_houston_spectral_upscale


2026-06-06 23:57:58,928 - Loaded houston (all): 15029 samples
2026-06-06 23:57:59,945 - [HyperSIGMA] SpecViT load summary: dropped=104, missing=16, unexpected=0, pos_embed=loaded
2026-06-06 23:57:59,946 - [HyperSIGMA] SpecViT missing keys (first 10): ['blocks.0.attn.sampling_offsets.weight', 'blocks.0.attn.sampling_offsets.bias', 'blocks.1.attn.sampling_offsets.weight', 'blocks.1.attn.sampling_offsets.bias', 'blocks.3.attn.sampling_offsets.weight', 'blocks.3.attn.sampling_offsets.bias', 'blocks.4.attn.sampling_offsets.weight', 'blocks.4.attn.sampling_offsets.bias', 'blocks.6.attn.sampling_offsets.weight', 'blocks.6.attn.sampling_offsets.bias']
2026-06-06 23:57:59,947 - [HyperSIGMA] Built branches: spat=False, spec=True, sem=False
2026-06-06 23:57:59,947 - [HyperSIGMA] Native-geometry ablation: input_fit=upscale (single-branch, unadapted)
2026-06-06 23:57:59,948 - [HyperSIGMA] SpecViT branch (SpecViT_fusion): cls_token=None, spec_embed=AdaptiveAvgPool1d(out=100), spat_map.weight.shape=(

## Step 5 - Summary table

Headline OA / AA / Kappa (primary distance) across all native-geometry runs.


In [ ]:
primary = base_eval_params.get("distance_metric", "euclidean")
print(f"Native-geometry ablation - {primary} OA / AA / Kappa (mean +/- CI95)\n")
print(f"{'dataset':8s} {'branch':9s} {'fit':8s} {'OA':>16s} {'AA':>16s} {'Kappa':>16s}")
print("-" * 70)
for r in runs:
    blk = r["results"].get(primary, {})
    oa, aa, kp = blk.get("OA", {}), blk.get("AA", {}), blk.get("Kappa", {})
    print(f"{r['dataset']:8s} {r['branch']:9s} {r['fit']:8s} "
          f"{oa.get('mean', float('nan')):>7.2f} +/-{oa.get('ci_95', 0):<5.2f} "
          f"{aa.get('mean', float('nan')):>7.2f} +/-{aa.get('ci_95', 0):<5.2f} "
          f"{kp.get('mean', float('nan')):>7.2f} +/-{kp.get('ci_95', 0):<5.2f}")
